<a href="https://colab.research.google.com/github/saimouzhang-research/Reuse-of-new-named-entity-task/blob/main/Entire%20script_Mar%2021%20with%20tokens_mask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1. Download / load data
# =========================
!wget https://cs.stanford.edu/~minalee/zip/chi2022-coauthor-v1.0.zip
!unzip -q chi2022-coauthor-v1.0.zip
!rm chi2022-coauthor-v1.0.zip


--2026-03-21 15:54:42--  https://cs.stanford.edu/~minalee/zip/chi2022-coauthor-v1.0.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49956179 (48M) [application/zip]
Saving to: ‘chi2022-coauthor-v1.0.zip’

chi2022-coauthor-v1 100%[===================>]  47.64M  22.7MB/s    in 2.1s    

2026-03-21 15:54:44 (22.7 MB/s) - ‘chi2022-coauthor-v1.0.zip’ saved [49956179/49956179]

replace coauthor-v1.0/e0435f4cf6fc435c872ffc5b66b66b0c.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [2]:
import os
import json
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
# =========================
# 2. Read session paths
# =========================
def find_writing_sessions(dataset_dir):
    return [
        os.path.join(dataset_dir, path)
        for path in os.listdir(dataset_dir)
        if path.endswith('jsonl')
    ]

def read_writing_session(path):
    events = []
    with open(path, 'r') as f:
        for event in f:
            events.append(json.loads(event))
    return events

dataset_dir = './coauthor-v1.0'
paths = find_writing_sessions(dataset_dir)

In [4]:
# =========================
# 3. Load metadata
# =========================
def sess2auth(zip_list):
    out_dict = {}
    for author, session in zip_list:
        if session not in out_dict:
            out_dict[session.strip()] = author.strip()
    return out_dict

df_a = pd.read_csv('/content/CoAuthor - Metadata & Survey - Metadata (argumentative).csv')
df_c = pd.read_csv('/content/CoAuthor - Metadata & Survey - Metadata (creative).csv')

session_id_col = 'session_id'
worker_id_col = 'worker_id'

df_a_list = list(zip(df_a[worker_id_col], df_a[session_id_col]))
df_c_list = list(zip(df_c[worker_id_col], df_c[session_id_col]))

sess_auth_dict = sess2auth(df_a_list + df_c_list)


In [5]:
# =========================
# 4. Reconstruct text + mask
# =========================
def apply_ops(doc, mask, ops, source):
    original_doc = doc
    original_mask = mask

    new_doc = ''
    new_mask = ''

    for op in ops:
        if 'retain' in op:
            num_char = op['retain']
            retain_doc = original_doc[:num_char]
            retain_mask = original_mask[:num_char]

            original_doc = original_doc[num_char:]
            original_mask = original_mask[num_char:]

            new_doc += retain_doc
            new_mask += retain_mask

        elif 'insert' in op:
            insert_doc = op['insert']

            if isinstance(insert_doc, dict):
                # skip non-text insertions
                continue

            insert_mask = 'A' * len(insert_doc) if source == 'api' else 'U' * len(insert_doc)

            new_doc += insert_doc
            new_mask += insert_mask

        elif 'delete' in op:
            num_char = op['delete']

            if original_doc:
                original_doc = original_doc[num_char:]
                original_mask = original_mask[num_char:]
            else:
                new_doc = new_doc[:-num_char]
                new_mask = new_mask[:-num_char]

    final_doc = new_doc + original_doc
    final_mask = new_mask + original_mask
    return final_doc, final_mask

def get_text_and_mask(events, event_id, remove_prompt=False):
    prompt = events[0]['currentDoc'].strip()

    text = prompt
    mask = 'P' * len(prompt)

    for event in events[:event_id]:
        if 'textDelta' not in event or 'ops' not in event['textDelta']:
            continue

        ops = event['textDelta']['ops']
        source = event.get('eventSource', 'user')
        text, mask = apply_ops(text, mask, ops, source)

    if remove_prompt and 'P' in mask:
        end_index = mask.rindex('P')
        text = text[end_index + 1:]
        mask = mask[end_index + 1:]

    return text, mask

In [6]:
# =========================
# 5. Prompt ID mapping
# =========================
prompt_shapeshifter = {''.join('A woman has'.split()): 'shapeshifter'}
prompt_reincarnation = {''.join('When you die,'.split()): 'reincarnation'}
prompt_mana = {''.join('Humans once wielded'.split()): 'mana'}
prompt_obama = {''.join("You're Barack Obama.".split()): 'obama'}
prompt_pig = {''.join('Once upon a'.split()): 'pig'}
prompt_mattdamon = {''.join('An alien has'.split()): 'mattdamon'}
prompt_sideeffect = {''.join("When you're 28,".split()): 'sideeffect'}
prompt_bee = {''.join("Your entire life,".split()): 'bee'}
prompt_dad = {''.join("All of the".split()): 'dad'}
prompt_isolation = {''.join("Following World War".split()): 'isolation'}
prompt_screen = {''.join('How Worried Should'.split()): 'screen'}
prompt_dating = {''.join('How Do You'.split()): 'dating'}
prompt_pads = {''.join('Should Schools Provide'.split()): 'pads'}
prompt_school = {''.join('What Are the'.split()): 'school'}
prompt_stereotype = {''.join('What Stereotypical Characters'.split()): 'stereotype'}
prompt_audiobook = {''.join('Is Listening to'.split()): 'audiobook'}
prompt_athletes = {''.join('Should College Athletes'.split()): 'athletes'}
prompt_extremesports = {''.join('Is It Selfish'.split()): 'extremesports'}
prompt_animal = {''.join('Is It Wrong'.split()): 'animal'}
prompt_news = {''.join("Are We Being".split()): 'news'}

prompt_dict = (
    prompt_isolation
    | prompt_dad
    | prompt_shapeshifter
    | prompt_reincarnation
    | prompt_mana
    | prompt_obama
    | prompt_pig
    | prompt_mattdamon
    | prompt_sideeffect
    | prompt_bee
    | prompt_screen
    | prompt_dating
    | prompt_pads
    | prompt_school
    | prompt_stereotype
    | prompt_audiobook
    | prompt_athletes
    | prompt_extremesports
    | prompt_animal
    | prompt_news
)

def get_prompt_id_from_text(text, prompt_dict=prompt_dict):
    prompt_first = ''.join(text.split()[:3])
    return prompt_dict.get(prompt_first, 'misc')


In [7]:
# =========================
# 6. Sentence-level masks
# =========================
def classify_sentences_with_mask_flat(text, mask):
    sentence_rows = []
    search_start = 0

    for sentence_id, sentence in enumerate(sent_tokenize(text.strip())):
        sentence = sentence.strip()
        if not sentence:
            continue

        start_idx = text.find(sentence, search_start)
        if start_idx == -1:
            start_idx = text.find(sentence)
        if start_idx == -1:
            print(f'Could not find sentence in text: {sentence}')
            continue

        end_idx = start_idx + len(sentence)
        sentence_mask = mask[start_idx:end_idx]

        chars = set(sentence_mask)
        if chars == {'A'}:
            sentence_author = 'api'
        elif chars == {'U'}:
            sentence_author = 'user'
        else:
            sentence_author = 'user_and_api'

        sentence_rows.append({
            'sentence_id': sentence_id,
            'sentence_text': sentence,
            'sentence_mask': sentence_mask,
            'sentence_author': sentence_author
        })

        search_start = end_idx

    return sentence_rows

In [8]:
# =========================
# 7. Build final dataframe
# =========================
rows = []

for path in paths:
    session_id = path.split('/')[2].split('.')[0].strip()
    author_id = sess_auth_dict.get(session_id, 'missing_info')

    events = read_writing_session(path)
    text, mask = get_text_and_mask(events, len(events), remove_prompt=False)
    prompt_id = get_prompt_id_from_text(text)
    sentence_with_masks = classify_sentences_with_mask_flat(text, mask)

    rows.append({
        'session_id': session_id,
        'author_id': author_id,
        'prompt_id': prompt_id,
        'text': text,
        'mask': mask,
        'sentence_with_masks': json.dumps(sentence_with_masks, ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)
df_out.to_csv('CoAuthor_session_text_masks.csv', index=False)

print(df_out.head())
print(df_out.columns)
print(df_out.shape)

                         session_id       author_id      prompt_id  \
0  69655d09325f45e6bc4dbe10afdad674  A2WGW5Y3ZFBDEC         animal   
1  96877ba434a04724a95095ad37efaf45    missing_info           misc   
2  ae8434c075224ebdaba8e215723c9bbb   A2W121DQXNQK1     stereotype   
3  200b5a62bded44f6ba8320c117bee58a    missing_info  reincarnation   
4  5a79775f3b054f82a6cb92319da17db5  A2EED3HLTA96CP     sideeffect   

                                                text  \
0  Is It Wrong to Focus on Animal Welfare When Hu...   
1  Yesterday, you were laying in a hospital bed, ...   
2  What Stereotypical Characters Make You Cringe?...   
3  When you die, you appear in a cinema with a nu...   
4  When you're 28, science discovers a drug that ...   

                                                mask  \
0  PPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPP...   
1  PUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU...   
2  PPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPP...   
3  PPPPPPPPPPPPPPP

In [9]:
print(df_a.columns)
print(df_c.columns)

Index(['timestamp', 'session_id', 'prompt_code', 'temperature',
       'frequency_penalty', 'num_query', 'num_selected', 'perc_selected',
       'num_cursor', 'time', 'num_event', 'written_by_human', 'X', 'X.1',
       'X.2', 'X.3', 'X.4', 'X.5', 'worker_id',
       'Is.English.your.first.language.',
       'How.qualified.do.you.think.you.are.to.write.about.this.topic...based.on.background.knowledge..understanding.of.prompts..etc...',
       'During.this.writing.session..the.suggestions.you.received.were.grammatically.correct..and.contributed.to.the.fluency.of.the.essay.',
       'During.this.writing.session..the.suggestions.helped.me.come.up.with.new.ideas.',
       'During.this.writing.session..I.felt.like.that.I.would.have.written.a..better..essay.if.I.wrote.the.essay.alone.',
       'During.this.writing.session..the.system.was.competent..having.expert.knowledge.and.ability.to.perform.a.task.successfully..in.writing.',
       'During.this.writing.session..the.system.was.capable.of.w

In [10]:
print(len(sess_auth_dict))

1252


In [11]:
print(paths[0])
print(session_id)

./coauthor-v1.0/69655d09325f45e6bc4dbe10afdad674.jsonl
ae3a87fc6c5e4c18897f71b93d670529


In [12]:
metadata_sessions = set(sess_auth_dict.keys())

file_sessions = set([
    path.split('/')[2].split('.')[0].strip()
    for path in paths
])

print("Total JSON sessions:", len(file_sessions))
print("Metadata sessions:", len(metadata_sessions))
print("Overlap:", len(file_sessions & metadata_sessions))

Total JSON sessions: 1447
Metadata sessions: 1252
Overlap: 1252


In [ ]:
# =========================
# 1. Setup
# =========================
import os
import json
import re
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================
# 2. Find and read sessions
# =========================
def find_writing_sessions(dataset_dir):
    return [
        os.path.join(dataset_dir, fname)
        for fname in os.listdir(dataset_dir)
        if fname.endswith(".jsonl")
    ]

def read_writing_session(path):
    events = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            events.append(json.loads(line))
    return events

# =========================
# 3. Metadata -> session_id : author_id
# =========================
def sess2auth(zip_list):
    out_dict = {}
    for author, session in zip_list:
        session = str(session).strip()
        author = str(author).strip()
        if session not in out_dict:
            out_dict[session] = author
    return out_dict

# Replace paths if needed
df_a = pd.read_excel('/content/CoAuthor - Metadata & Survey - Metadata (argumentative)_.xlsx')
df_c = pd.read_excel('/content/CoAuthor - Metadata & Survey - Metadata (creative)_.xlsx')

# Adjust these column names if your files use different names
session_id_col = 'session_id'
worker_id_col = 'worker_id'

df_a_list = list(zip(df_a[worker_id_col], df_a[session_id_col]))
df_c_list = list(zip(df_c[worker_id_col], df_c[session_id_col]))

sess_auth_dict = sess2auth(df_a_list + df_c_list)

# =========================
# 4. Quill-style text + mask reconstruction
# =========================
def apply_ops(doc, mask, ops, source):
    """
    Apply Quill Delta ops to the current document and provenance mask.
    Mask codes:
      P = prompt
      U = user
      A = api
    """
    original_doc = doc
    original_mask = mask

    new_doc = ''
    new_mask = ''

    for op in ops:
        if 'retain' in op:
            n = op['retain']
            new_doc += original_doc[:n]
            new_mask += original_mask[:n]
            original_doc = original_doc[n:]
            original_mask = original_mask[n:]

        elif 'insert' in op:
            inserted = op['insert']

            # Skip embeds / non-text objects
            if isinstance(inserted, dict):
                continue

            inserted = str(inserted)
            inserted_mask = ('A' if source == 'api' else 'U') * len(inserted)

            new_doc += inserted
            new_mask += inserted_mask

        elif 'delete' in op:
            n = op['delete']

            # delete from remaining original content
            if len(original_doc) >= n:
                original_doc = original_doc[n:]
                original_mask = original_mask[n:]
            else:
                # fallback in case deletion spills backward
                overflow = n - len(original_doc)
                original_doc = ''
                original_mask = ''
                if overflow > 0:
                    new_doc = new_doc[:-overflow] if overflow <= len(new_doc) else ''
                    new_mask = new_mask[:-overflow] if overflow <= len(new_mask) else ''

    final_doc = new_doc + original_doc
    final_mask = new_mask + original_mask
    return final_doc, final_mask

def get_text_and_mask(events, event_id, remove_prompt=False):
    prompt = events[0]['currentDoc'].strip()

    text = prompt
    mask = 'P' * len(prompt)

    for event in events[:event_id]:
        if 'textDelta' not in event:
            continue
        if 'ops' not in event['textDelta']:
            continue

        ops = event['textDelta']['ops']
        source = event.get('eventSource', 'user')
        text, mask = apply_ops(text, mask, ops, source)

    if remove_prompt and 'P' in mask:
        end_prompt = mask.rfind('P')
        text = text[end_prompt + 1:]
        mask = mask[end_prompt + 1:]

    return text, mask

# =========================
# 5. Prompt mapping
# =========================
prompt_dict = {
    ''.join('A woman has'.split()): 'shapeshifter',
    ''.join('When you die,'.split()): 'reincarnation',
    ''.join('Humans once wielded'.split()): 'mana',
    ''.join("You're Barack Obama.".split()): 'obama',
    ''.join('Once upon a'.split()): 'pig',
    ''.join('An alien has'.split()): 'mattdamon',
    ''.join("When you're 28,".split()): 'sideeffect',
    ''.join("Your entire life,".split()): 'bee',
    ''.join("All of the".split()): 'dad',
    ''.join("Following World War".split()): 'isolation',
    ''.join('How Worried Should'.split()): 'screen',
    ''.join('How Do You'.split()): 'dating',
    ''.join('Should Schools Provide'.split()): 'pads',
    ''.join('What Are the'.split()): 'school',
    ''.join('What Stereotypical Characters'.split()): 'stereotype',
    ''.join('Is Listening to'.split()): 'audiobook',
    ''.join('Should College Athletes'.split()): 'athletes',
    ''.join('Is It Selfish'.split()): 'extremesports',
    ''.join('Is It Wrong'.split()): 'animal',
    ''.join("Are We Being".split()): 'news'
}

def get_prompt_id_from_text(text, prompt_dict=prompt_dict):
    prompt_first = ''.join(text.split()[:3])
    return prompt_dict.get(prompt_first, 'misc')

# =========================
# 6. Sentence-level mask mapping
# =========================
def label_from_mask(mask_string):
    chars = set(mask_string)
    if chars == {'A'}:
        return 'api'
    elif chars == {'U'}:
        return 'user'
    elif chars == {'P'}:
        return 'prompt'
    elif chars <= {'A', 'U'}:
        return 'user_and_api'
    elif chars <= {'P', 'U'}:
        return 'prompt_and_user'
    elif chars <= {'P', 'A'}:
        return 'prompt_and_api'
    else:
        return 'mixed'

def classify_sentences_with_mask_flat(text, mask):
    sentence_rows = []
    search_start = 0

    for sentence_id, sentence in enumerate(sent_tokenize(text.strip())):
        sentence = sentence.strip()
        if not sentence:
            continue

        start_idx = text.find(sentence, search_start)
        if start_idx == -1:
            start_idx = text.find(sentence)
        if start_idx == -1:
            print(f'Could not find sentence in text: {sentence}')
            continue

        end_idx = start_idx + len(sentence)
        sentence_mask = mask[start_idx:end_idx]

        sentence_rows.append({
            'sentence_id': sentence_id,
            'sentence_text': sentence,
            'start_char': start_idx,
            'end_char': end_idx,
            'sentence_mask': sentence_mask,
            'sentence_author': label_from_mask(sentence_mask)
        })

        search_start = end_idx

    return sentence_rows

# =========================
# 7. Token-level mask mapping
# =========================
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def classify_tokens_with_mask(text, mask):
    token_rows = []

    for token_id, match in enumerate(TOKEN_PATTERN.finditer(text)):
        token_text = match.group()
        start_idx = match.start()
        end_idx = match.end()

        token_mask = mask[start_idx:end_idx]

        token_rows.append({
            'token_id': token_id,
            'token_text': token_text,
            'start_char': start_idx,
            'end_char': end_idx,
            'token_mask': token_mask,
            'token_author': label_from_mask(token_mask)
        })

    return token_rows

# =========================
# 8. Build final dataframe
# =========================
dataset_dir = './coauthor-v1.0'   # adjust if needed
paths = find_writing_sessions(dataset_dir)

rows = []

for path in paths:
    session_id = os.path.basename(path).split('.')[0].strip()
    author_id = sess_auth_dict.get(session_id, 'missing_info')

    events = read_writing_session(path)
    text, mask = get_text_and_mask(events, len(events), remove_prompt=False)
    prompt_id = get_prompt_id_from_text(text)

    sentence_with_masks = classify_sentences_with_mask_flat(text, mask)
    token_with_masks = classify_tokens_with_mask(text, mask)

    rows.append({
        'session_id': session_id,
        'author_id': author_id,
        'prompt_id': prompt_id,
        'text': text,
        'mask': mask,
        'sentence_with_masks': json.dumps(sentence_with_masks, ensure_ascii=False),
        'token_with_masks': json.dumps(token_with_masks, ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)
df_out.to_csv('CoAuthor_session_text_masks_tokens.csv', index=False)

print(df_out.head())
print(df_out.shape)
print(df_out.columns)